[![Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/drigaa/rEVA_CE4IB/HEAD?urlpath=%2Fdoc%2Ftree%2Fcasestudy.ipynb)

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from tqdm import tqdm

import dm4bem as db

In [ ]:
from settings_simulation import *

### Thermal Network

![thermal-network](images/Screenshot%202026-10-29%20093356.png)

### Material properties and geometry

In [ ]:
number_of_nodes = 10
number_of_branches = 14

In [ ]:
air = {
    'Density': 1.2,
    'Specific heat': 1000
}

# convection coefficients, W/(m²·K)
h = pd.DataFrame([{'in': 4., 'out': 10.}], index=['h'])

In [ ]:
l = 3

concrete = {
    'Conductivity': 1.400,
    'Density': 2300.0,
    'Specific heat': 880,
    'Width': 0.15,
    'Surface': l**2
}

insulation = {
    'Conductivity': 0.040,
    'Density': 16.0,
    'Specific heat': 1210,
    'Width': 0.25,
    'Surface': l**2
}

glass = {
    'U-Value': 0.6,
    'Surface': l**2
}

wall = pd.DataFrame.from_dict(
    {
        'Layer_out': insulation,
        'Layer_in': concrete,
        'window': glass
    },
    orient='index')

wall

### System of algebraic-differential equations DAE
#### Temp nodes and flow branches

In [ ]:
θ = [f'θ{i}' for i in range(number_of_nodes)]
q = [f'q{i}' for i in range(number_of_branches)]

#### Incidence matrix $A$


In [ ]:
A = np.zeros((number_of_branches, number_of_nodes))

A[0, 0] = 1
A[1, 0], A[1, 1] = -1, 1
A[2, 1], A[2, 2] = -1, 1
A[3, 2], A[3, 3] = -1, 1
A[4, 3], A[4, 4] = -1, 1
A[5, 4], A[5, 5] = -1, 1
A[6, 6], A[6, 5] = -1, 1
A[7, 7], A[7, 6] = -1, 1
A[8, 8], A[8, 7] = -1, 1
A[9, 9], A[9, 8] = -1, 1
A[10, 9] = 1
A[11, 5] = 1
A[12, 9] = 1
A[13, 5] = 1

A = pd.DataFrame(A, index=q, columns=θ)

#### Conductance Matrix $G$

In [ ]:
## P-controler gain
Kp = 0

G = np.zeros(number_of_branches)

G[[1, 2]] = (wall["Conductivity"] / (wall["Width"] / 2) * wall["Surface"]).Layer_out
G[[3, 4, 7 ,8]] =  (wall["Conductivity"] / (wall["Width"] / 2) * wall["Surface"]).Layer_in

G[[0]] = h['out'] * wall["Surface"].Layer_out
G[[5, 6, 9]] = h['in'] * wall["Surface"].Layer_in

G[[12, 13]] = (wall['U-Value'] * wall["Surface"]).window

G[[10, 11]] = Kp
G = pd.Series(G, index=q)
G

#### Capacity Matrix $C$

In [ ]:
C = np.zeros(number_of_nodes)

C[1] = (wall['Width'] * wall['Surface'] * wall['Density'] * wall["Specific heat"]).Layer_out
C[[3, 7]] = (wall['Width'] * wall['Surface'] * wall['Density'] * wall["Specific heat"]).Layer_in
C[[5, 9]] = air["Density"] * l**3 * air["Specific heat"]
C = pd.Series(C, index=θ)
C

#### Temperatur source vector $b$

In [ ]:
b = pd.Series(np.zeros(number_of_branches), index=q)
b[[0, 12, 13]] = "T_0"
b[[10, 11]] = "Ti_sp"
b

#### Heat flow source vector $f$

In [ ]:
f = pd.Series(np.zeros(number_of_nodes), index=θ)
f[[0]] = 'Φo'
f[[5, 9]] = 'Qa'
f

#### Output vector $y$

In [ ]:
y = pd.Series(np.zeros(number_of_nodes), index=θ)
y[[5, 9]] = 1
y

In [ ]:
TC = {"A": A,
      "G": G,
      "C": C,
      "b": b,
      "f": f,
      "y": y}

### Steady state analysis from DAE

In [ ]:
if controller:
    TC["G"][["q10", "q11"]] = Kp_value

In [ ]:
# State-space
[As, Bs, Cs, Ds, us] = db.tc2ss(TC)

##### Overwrite $b$ and $f$ vector for free floating building (no controller)

In [ ]:
b_ss = np.zeros(number_of_branches)
b_ss[[0, 12, 13]] = To_ss
b_ss[[10, 11]] = Ti_sp_ss

f_ss = np.zeros(number_of_nodes)

A = TC['A']
G = TC['G']
diag_G = pd.DataFrame(np.diag(G), index=G.index, columns=G.index)

θss = np.linalg.inv(A.T @ diag_G @ A) @ (A.T @ diag_G @ b_ss + f_ss)
print("Results free floating building")
print(f'θss = {np.around(θss, 2)} °C')

In [ ]:
b_ss = np.zeros(number_of_branches)
b_ss[[0, 12, 13]] = To_ss
b_ss[[10, 11]] = Ti_sp_ss

f_ss = np.zeros(number_of_nodes)
f_ss[[5, 9]] = Qa_ss

A = TC['A']
G = TC['G']
diag_G = pd.DataFrame(np.diag(G), index=G.index, columns=G.index)

θss_Q = np.linalg.inv(A.T @ diag_G @ A) @ (A.T @ diag_G @ b_ss + f_ss)
print("Results with auxiliary heat source 50 Watts on node (room) 5 and 9")
print(f'θss_Q = {np.around(θss_Q, 2)} °C')

### Steady State analysis from state-space represintation

In [ ]:
# Original represantation
us

In [ ]:
bT = np.array([To_ss, Ti_sp_ss, Ti_sp_ss, To_ss, To_ss])
fQ = np.array([0, Qa_ss, Qa_ss])
uss = np.hstack([bT, fQ])
uss = pd.Series(uss, index=us.index)
uss

In [ ]:
inv_As = pd.DataFrame(np.linalg.inv(As),
                      columns=As.index, index=As.index)
yss = (-Cs @ inv_As @ Bs + Ds) @ uss

print(f"Node/Room {yss.index[0]}: {yss[0]:.2f}")
print(f"Node/Room {yss.index[1]}: {yss[1]:.2f}")

In [ ]:
print(f"Error between DAE and state-space\n{abs(yss - θss_Q[[5, 9]])}")

### Eigenvalue analysis
#### Time step

In [ ]:
λ = np.linalg.eig(As)[0]

# time step
Δtmax = 2 * min(-1. / λ)    # max time step for stability of Euler explicit
db.print_rounded_time('Δtmax', Δtmax)

if imposed_time_step:
    dt = Δt
else:
    dt = db.round_time(Δtmax)

db.print_rounded_time('dt', dt)
print(f"dt = {dt:.0f} s")

#### Settling time

In [ ]:
# settling time
t_settle = 4 * max(-1 / λ)
db.print_rounded_time('t_settle', t_settle)

# duration: next multiple of 3600 s that is larger than t_settle
duration = np.ceil(t_settle / 3600) * 3600
db.print_rounded_time('duration', duration)

### Step response to outdoor temperature
#### Input vector

In [ ]:
filename = Path("weather_data/Hombrechtikon/CHE_ZH_Hombrechtikom.068510_TMYx.2011-2025.epw")
[data, meta] = db.read_epw(filename, coerce_year=None)
weather = data[["temp_air", "dir_n_rad", "dif_h_rad"]]

In [ ]:
weather.index = weather.index.map(lambda t: t.replace(year=2000))
weather = weather.loc[start_date:end_date]

In [ ]:
# Create input_data_set
n = weather.shape[0]

To = weather["temp_air"]         # outdoor temperature
Ti_sp = Ti_sp_ss * np.ones(n)   # indoor temperature set point
Φa = 0 * np.ones(n)             # solar radiation absorbed by the glass
Φo = Φi = Φa
Qa = Qa_ss * np.ones(n)         # auxiliary heat sources and solar radiation

data = {'T_0': To, 'Ti_sp': Ti_sp, 'Φo': Φo, 'Φi': Φi, 'Qa': Qa, 'Φa': Φa}
input_data_set = pd.DataFrame(data, index=weather.index).resample(str(dt) + 'S').interpolate(method="linear")

u = db.inputs_in_time(us, input_data_set)

#### Time integration

In [ ]:
# Initial conditions
θ_exp = pd.DataFrame(index=u.index)     # empty df with index for explicit Euler
θ_imp = pd.DataFrame(index=u.index)     # empty df with index for implicit Euler

θ_exp[As.columns] = temp_init     # fill θ for Euler explicit with initial values θ0
θ_imp[As.columns] = temp_init    # fill θ for Euler implicit with initial values θ0

I = np.eye(As.shape[0])

for k in tqdm(range(u.shape[0] - 1)):
    θ_exp.iloc[k + 1] = (I + dt * As) @ θ_exp.iloc[k] + dt * Bs @ u.iloc[k]
    θ_imp.iloc[k + 1] = np.linalg.inv(I - dt * As) @ (θ_imp.iloc[k] + dt * Bs @ u.iloc[k])

# outputs
y_exp = (Cs @ θ_exp.T + Ds @  u.T).T
y_imp = (Cs @ θ_imp.T + Ds @  u.T).T

In [ ]:
# plot results
y = pd.concat([y_exp.iloc[:,0], y_imp.iloc[:,0]], axis=1, keys=['Explicit', 'Implicit'])
# Flatten the two-level column labels into a single level
y.columns = y.columns.get_level_values(0)

ax = y.plot()
ax.set_xlabel('Time')
ax.set_ylabel('Indoor temperature, $\\theta_i$ / °C')
ax.set_title(f'Node/Room 5\nTime step: $dt$ = {dt:.0f} s; $dt_{{max}}$ = {Δtmax:.0f} s')
plt.show()

In [ ]:
# plot results
y = pd.concat([y_exp.iloc[:,1], y_imp.iloc[:,1]], axis=1, keys=['Explicit', 'Implicit'])
# Flatten the two-level column labels into a single level
y.columns = y.columns.get_level_values(0)

ax = y.plot()
ax.set_xlabel('Time')
ax.set_ylabel('Indoor temperature, $\\theta_i$ / °C')
ax.set_title(f'Node/Room 9\nTime step: $dt$ = {dt:.0f} s; $dt_{{ max}}$ = {Δtmax:.0f} s')
plt.show()